<a href="https://colab.research.google.com/github/Thorfast191/qAIR/blob/main/qAIR_V6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
# !pip install pennylane pennylane-lightning-gpu --upgrade
# !pip install custatevec-cu12 cuquantum-cu12

# Imports

In [29]:
# =========================
# IMPORTS
# =========================
import torch
import torch.nn as nn
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import time
import math

# =========================
# CONFIG
# =========================
torch.manual_seed(42)

n_qubits = 6
n_layers = 4
input_dim = 16
hidden_dim = 32
vocab_size = 10
batch_size = 16
epochs = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# =========================
# QUANTUM DEVICE
# =========================
dev = qml.device("lightning.gpu", wires=n_qubits)

Using device: cuda


# QRL

In [30]:
@qml.qnode(dev, interface="torch", diff_method="parameter-shift")
def qrl_circuit(inputs, weights):

    for l in range(n_layers):

        # Data re-uploading
        for i in range(n_qubits):
            qml.RY(inputs[i], wires=i)
            qml.RZ(inputs[i], wires=i)

        # Variational block
        for i in range(n_qubits):
            qml.RX(weights[l, i, 0], wires=i)
            qml.RZ(weights[l, i, 1], wires=i)

        # Ring entanglement
        for i in range(n_qubits):
            qml.CNOT(wires=[i, (i + 1) % n_qubits])

    return qml.probs(wires=range(n_qubits))

# Convert QNode → Torch Layer

In [31]:
weight_shapes = {"weights": (n_layers, n_qubits, 2)}
qlayer = qml.qnn.TorchLayer(qrl_circuit, weight_shapes)

# Classical Baseline Model

In [32]:
class ClassicalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

    def forward(self, x):
        return self.net(x.to(device))

# Hybrid Quantum Model

In [33]:
class HybridQuantumModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_qubits)
        )

        self.q_layer = qlayer
        self.q_out_dim = 2 ** n_qubits

        self.decoder = nn.Linear(self.q_out_dim, vocab_size)

        self.classical_head = nn.Linear(hidden_dim, vocab_size)

        self.alpha = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):

        x = x.to(device)

        # classical path
        h_mid = torch.relu(self.encoder[0](x))
        classical_logits = self.classical_head(h_mid)

        # quantum path
        h = self.encoder(x)
        h = torch.tanh(h) * math.pi
        h = h + 0.005 * torch.randn_like(h)

        q_out = torch.stack([self.q_layer(h_i) for h_i in h])
        quantum_logits = self.decoder(q_out)

        logits = self.alpha * quantum_logits + (1 - self.alpha) * classical_logits

        return logits

# Dataset

In [34]:
x = torch.randn(512, input_dim)
y = torch.randint(0, vocab_size, (512,))

dataset = TensorDataset(x, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Training Loop

In [35]:
def train(model, dataloader, optimizer, loss_fn, epochs):

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")

        for xb, yb in pbar:

            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()

            logits = model(xb)
            loss = loss_fn(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        print(f"\nEpoch {epoch+1} Avg Loss: {epoch_loss/len(dataloader):.4f}")

# Evaluation Function

In [36]:
def evaluate(model, dataloader):

    model.eval()

    correct = 0
    total = 0

    start_time = time.time()

    with torch.no_grad():
        for xb, yb in dataloader:

            xb, yb = xb.to(device), yb.to(device)

            logits = model(xb)
            preds = logits.argmax(dim=1)

            correct += (preds == yb).sum().item()
            total += yb.size(0)

    end_time = time.time()

    accuracy = correct / total
    time_per_sample = (end_time - start_time) / total

    return accuracy, time_per_sample

# Run

In [37]:
loss_fn = nn.CrossEntropyLoss()

# ---- Classical ----
print("\n🧠 Training Classical Model...\n")
classical_model = ClassicalModel().to(device)
optimizer_c = torch.optim.Adam(classical_model.parameters(), lr=1e-3)

train(classical_model, dataloader, optimizer_c, loss_fn, epochs)
acc_c, t_c = evaluate(classical_model, dataloader)

# ---- Quantum ----
print("\n⚛️ Training Hybrid Quantum Model...\n")
quantum_model = HybridQuantumModel().to(device)
optimizer_q = torch.optim.Adam(quantum_model.parameters(), lr=1e-3)

train(quantum_model, dataloader, optimizer_q, loss_fn, epochs)
acc_q, t_q = evaluate(quantum_model, dataloader)

# =========================
# RESULTS
# =========================
print("\n📊 FINAL COMPARISON\n")

print(f"Classical Accuracy: {acc_c*100:.2f}%")
print(f"Quantum Accuracy:   {acc_q*100:.2f}%\n")

print(f"Classical Time/sample: {t_c:.6f} sec")
print(f"Quantum Time/sample:   {t_q:.6f} sec")


🧠 Training Classical Model...



Epoch 1/5: 100%|██████████| 32/32 [00:00<00:00, 320.60it/s, loss=2.3282]



Epoch 1 Avg Loss: 2.3198


Epoch 2/5: 100%|██████████| 32/32 [00:00<00:00, 311.45it/s, loss=2.3347]



Epoch 2 Avg Loss: 2.2972


...


Epoch 3 Avg Loss: 2.2890


Epoch 4/5: 100%|██████████| 32/32 [14:11<00:00, 26.59s/it, loss=2.2342]



Epoch 4 Avg Loss: 2.2832


Epoch 5/5: 100%|██████████| 32/32 [14:07<00:00, 26.47s/it, loss=2.2437]



Epoch 5 Avg Loss: 2.2766

📊 FINAL COMPARISON

Classical Accuracy: 16.99%
Quantum Accuracy:   16.02%

Classical Time/sample: 0.000039 sec
Quantum Time/sample:   0.011129 sec
